# POC — Ingestão de CSV para MySQL
### Tabela: `tb_dados_brutos`
---
**Stack:** Python · Pandas · SQLAlchemy · mysql-connector-python  
**Pré-requisitos:**
```bash
pip install pandas sqlalchemy mysql-connector-python
```

## 1. Imports

In [1]:
import pandas as pd
from sqlalchemy import create_engine, text
import time

## 2. Configurações de conexão

In [2]:
# ── Configurações ──────────────────────────────────────────────
DB_HOST     = 'localhost'
DB_PORT     = 3306
DB_USER     = 'root'          # altere para seu usuário
DB_PASSWORD = '123456'     # altere para sua senha
DB_NAME     = 'db_call_center'        # altere se necessário

from pathlib import Path

CSV_PATH = (
    Path.cwd()          # D:\Git-Projetos\Python\POC-Call-Center\notebooks
    .parent             # D:\Git-Projetos\Python\POC-Call-Center
    .parent             # D:\Git-Projetos\Python
    .parent             # D:\Git-Projetos
    .parent             # D:\                          ← raiz comum
    / 'Users' / 'rtoni' / 'OneDrive' / 'Git-Dados' / 'POC-Call-Center'
    / 'tb_ligacao_202602271123.csv'
    # / 'Tabela_DDD.csv'
)
TABLE_NAME  = 'tb_dados_bruto'
# TABLE_NAME  = 'tb_ddd_brasil'
SEPARATOR   = ';'             # separador do CSV
CHUNK_SIZE  = 500             # linhas por lote (ajuste conforme volume)
# ───────────────────────────────────────────────────────────────

## 3. Criando a engine de conexão

In [3]:
connection_string = (
    f"mysql+mysqlconnector://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}?charset=utf8mb4"
)

engine = create_engine(connection_string, echo=False)

# Teste de conexão
with engine.connect() as conn:
    result = conn.execute(text("SELECT 'Conexão OK!' AS status"))
    print(result.fetchone()[0])

Conexão OK!


## 4. Leitura e validação do CSV

In [4]:
df = pd.read_csv(
    CSV_PATH,
    sep=SEPARATOR,
    dtype=str,            # lê tudo como texto (espelha VARCHAR no MySQL)
    keep_default_na=False # mantém campos vazios como string vazia
)

# Normaliza nomes de coluna (remove espaços extras, se houver)
df.columns = df.columns.str.strip()

print(f"✅ Arquivo lido com sucesso!")
print(f"   Linhas   : {len(df):,}")
print(f"   Colunas  : {len(df.columns)}")
print()
df.head(3)

✅ Arquivo lido com sucesso!
   Linhas   : 346,906
   Colunas  : 44



,id_ligacao,id_dsc_gravacao,id_dsc_ticket,id_lead,nu_dsc_telefone_to,nm_gp_rota,dh_inicio,dh_fim,dh_inicio_tabulacao,qt_tempo_duracao,...,bl_venda,bl_agendamento,bl_agendamento_pessoal,bl_telefone_finalizado,bl_finaliza_lead,bl_pre_venda,bl_recusa,bl_improdutivo,bl_auditor,bl_auditoria_backoffice
0,339155614,,,42868660,4619*****77,XPTO,2026-02-02 11:34:16.000,2026-02-02 11:34:21.410,2026-02-02 11:34:21.410,5,...,0,0,0,0,0,0,0,0,0,0
1,339155765,,,42872159,4379*****66,XPTO,2026-02-02 11:34:16.000,2026-02-02 11:34:31.153,2026-02-02 11:34:31.153,15,...,0,0,0,0,0,0,0,0,0,0
2,339155758,,,42865513,4319*****31,XPTO,2026-02-02 11:34:16.000,2026-02-02 11:34:31.153,2026-02-02 11:34:31.153,15,...,0,0,0,0,0,0,0,0,0,0


## 5. Validação das colunas x tabela MySQL

In [5]:
EXPECTED_COLUMNS = [
    # 'Prefixo', 'Estado','Cidades_Regioes'
    'id_ligacao', 'id_dsc_gravacao', 'id_dsc_ticket', 'id_lead',
    'nu_dsc_telefone_to', 'nm_gp_rota', 'dh_inicio', 'dh_fim',
    'dh_inicio_tabulacao', 'qt_tempo_duracao', 'qt_tempo_falando',
    'qt_tempo_tabulando', 'qt_aguardando_chamada', 'id_dsc_contato',
    'tp_dsc_contato', 'id_dsc_campanha', 'nm_dsc_campanha', 'id_dsc_lista',
    'nm_dsc_lista', 'id_dsc_mailing', 'nm_dsc_mailing', 'id_dsc_status',
    'nm_dsc_status', 'nm_dsc_grupo', 'nm_sub_grupo', 'id_dsc_tabulacao',
    'nm_dsc_tabulacao', 'id_supervisor', 'nm_supervisor', 'id_dsc_usuario',
    'nm_dsc_usuario', 'bl_atendido', 'bl_abordagem', 'bl_target', 'bl_venda',
    'bl_agendamento', 'bl_agendamento_pessoal', 'bl_telefone_finalizado',
    'bl_finaliza_lead', 'bl_pre_venda', 'bl_recusa', 'bl_improdutivo',
    'bl_auditor', 'bl_auditoria_backoffice'
]

missing = set(EXPECTED_COLUMNS) - set(df.columns)
extra   = set(df.columns) - set(EXPECTED_COLUMNS)

if missing:
    print(f"⚠️  Colunas FALTANDO no CSV : {missing}")
if extra:
    print(f"⚠️  Colunas EXTRAS no CSV  : {extra}")
if not missing and not extra:
    print("✅ Colunas validadas — CSV compatível com a tabela MySQL!")

✅ Colunas validadas — CSV compatível com a tabela MySQL!


## 6. Ingestão por lotes (chunked insert)

In [6]:
total_rows   = len(df)
total_chunks = (total_rows // CHUNK_SIZE) + (1 if total_rows % CHUNK_SIZE else 0)
rows_inserted = 0
start_time   = time.time()

print(f"🚀 Iniciando ingestão — {total_rows:,} linhas em {total_chunks} lote(s)...\n")

for i, chunk_start in enumerate(range(0, total_rows, CHUNK_SIZE), start=1):
    chunk = df.iloc[chunk_start : chunk_start + CHUNK_SIZE]

    chunk.to_sql(
        name      = TABLE_NAME,
        con       = engine,
        if_exists = 'append',   # nunca recria a tabela
        index     = False,
        method    = 'multi'     # insert em bloco (mais rápido)
    )

    rows_inserted += len(chunk)
    pct = (rows_inserted / total_rows) * 100
    print(f"   Lote {i:>3}/{total_chunks} — {rows_inserted:>6,}/{total_rows:,} linhas inseridas ({pct:.1f}%)")

elapsed = time.time() - start_time
print(f"\n✅ Ingestão concluída em {elapsed:.2f}s — {rows_inserted:,} linhas gravadas em `{TABLE_NAME}`")

🚀 Iniciando ingestão — 346,906 linhas em 694 lote(s)...

   Lote   1/694 —    500/346,906 linhas inseridas (0.1%)
   Lote   2/694 —  1,000/346,906 linhas inseridas (0.3%)
   Lote   3/694 —  1,500/346,906 linhas inseridas (0.4%)
   Lote   4/694 —  2,000/346,906 linhas inseridas (0.6%)
   Lote   5/694 —  2,500/346,906 linhas inseridas (0.7%)
   Lote   6/694 —  3,000/346,906 linhas inseridas (0.9%)
   Lote   7/694 —  3,500/346,906 linhas inseridas (1.0%)
   Lote   8/694 —  4,000/346,906 linhas inseridas (1.2%)
   Lote   9/694 —  4,500/346,906 linhas inseridas (1.3%)
   Lote  10/694 —  5,000/346,906 linhas inseridas (1.4%)
   Lote  11/694 —  5,500/346,906 linhas inseridas (1.6%)
   Lote  12/694 —  6,000/346,906 linhas inseridas (1.7%)
   Lote  13/694 —  6,500/346,906 linhas inseridas (1.9%)
   Lote  14/694 —  7,000/346,906 linhas inseridas (2.0%)
   Lote  15/694 —  7,500/346,906 linhas inseridas (2.2%)
   Lote  16/694 —  8,000/346,906 linhas inseridas (2.3%)
   Lote  17/694 —  8,500/346,90

## 7. Validação pós-carga

In [7]:
with engine.connect() as conn:
    count_db  = conn.execute(text(f"SELECT COUNT(*) FROM {TABLE_NAME}")).scalar()
    count_csv = len(df)

print(f"📊 Linhas no CSV           : {count_csv:,}")
print(f"📊 Linhas no MySQL         : {count_db:,}")

if count_db == count_csv:
    print("✅ Contagem OK — todos os registros foram inseridos!")
else:
    diff = count_csv - count_db
    print(f"⚠️  Divergência de {diff:,} linhas — verifique os logs.")

📊 Linhas no CSV           : 346,906
📊 Linhas no MySQL         : 346,906
✅ Contagem OK — todos os registros foram inseridos!


## 8. Preview dos dados no banco

In [8]:
df_check = pd.read_sql(
    f"SELECT * FROM {TABLE_NAME} LIMIT 5",
    con=engine
)

print(f"Preview — primeiras 5 linhas de `{TABLE_NAME}`:")
df_check

Preview — primeiras 5 linhas de `tb_dados_bruto`:


,id_ligacao,id_dsc_gravacao,id_dsc_ticket,id_lead,nu_dsc_telefone_to,nm_gp_rota,dh_inicio,dh_fim,dh_inicio_tabulacao,qt_tempo_duracao,...,bl_venda,bl_agendamento,bl_agendamento_pessoal,bl_telefone_finalizado,bl_finaliza_lead,bl_pre_venda,bl_recusa,bl_improdutivo,bl_auditor,bl_auditoria_backoffice
0,339155614,,,42868660,4619*****77,XPTO,2026-02-02 11:34:16.000,2026-02-02 11:34:21.410,2026-02-02 11:34:21.410,5,...,0,0,0,0,0,0,0,0,0,0
1,339155765,,,42872159,4379*****66,XPTO,2026-02-02 11:34:16.000,2026-02-02 11:34:31.153,2026-02-02 11:34:31.153,15,...,0,0,0,0,0,0,0,0,0,0
2,339155758,,,42865513,4319*****31,XPTO,2026-02-02 11:34:16.000,2026-02-02 11:34:31.153,2026-02-02 11:34:31.153,15,...,0,0,0,0,0,0,0,0,0,0
3,339155762,,,42867578,4429*****52,XPTO,2026-02-02 11:34:16.000,2026-02-02 11:34:31.147,2026-02-02 11:34:31.147,15,...,0,0,0,0,0,0,0,0,0,0
4,339155613,,,42868318,4839*****84,XPTO,2026-02-02 11:34:16.000,2026-02-02 11:34:21.257,2026-02-02 11:34:21.257,5,...,0,0,0,0,0,0,0,0,0,0
